# Forecast Volatility
> Measure how volatile a model's forecasts are across successive runs, and whether its revisions were worth making.

## Motivation

Accuracy is usually the headline metric for a forecasting model, but it is not the only thing that matters in production.

A multi-horizon system running on a schedule produces several overlapping forecasts for the same future target date, one from each **forecast creation date** (FCD), as more data becomes available. That sequence of updates is a **forecast revision**, and how consistent or erratic those revisions are is what we call **forecast volatility**.

Consider an electrical grid operator using load forecasts to plan power supply. A forecast that revises from 45 GW to 65 GW ahead of a heat wave is a *useful* revision: it tells operators to activate reserve plants. But forecasts that jump around erratically between FCDs, without new information justifying the change, undermine trust and complicate planning. The goal is not to eliminate revisions. It is to distinguish benign, informative revisions from excessive, erratic ones.

Accuracy metrics cannot make that distinction: metrics such as `MAE` and `sCRPS` score each forecast against the truth and average, which discards the relationship *between* forecasts of the same date. Two models can post identical accuracy while one is steady and the other thrashes.

This tutorial introduces the two forecast volatility metrics in `tsforge.metrics.stability_metrics` and applies them to two hand-built forecasters that accuracy metrics cannot tell apart.

## The metrics

Both metrics compare the two forecasts that land on the **same target date** from consecutive FCDs. Indexing by series $b$, FCD $t$ and horizon step $h$, those are:

- $\hat{\mathbf{y}}_{b,t,h+1}$, issued at FCD $t$, looking $h+1$ steps ahead
- $\hat{\mathbf{y}}_{b,t+1,h}$, issued one FCD later, looking $h$ steps ahead

The second is a revision of the first.

### Symmetric Forecast Percentage Change (sFPC)

sFPC measures the relative change in predicted quantiles across consecutive FCDs, giving a quantitative view of the forecast revision rate:

$$\mathrm{sFPC}_{q} = \frac{200}{B \times T \times H} \sum_{b,t,h} \frac{|\hat{Y}^{(q)}_{b,t+1,h}-\hat{Y}^{(q)}_{b,t,h+1}|}{|\hat{Y}^{(q)}_{b,t+1,h}| + |\hat{Y}^{(q)}_{b,t,h+1}|}$$

Inspired by sMAPE, the denominator is symmetric in the two forecasts. This keeps the metric well behaved when predicted values are small and avoids the division-by-zero problems common to traditional percentage-based metrics.

### Scaled Excess Volatility (sEV)

sFPC treats every revision as equally undesirable, even ones that clearly improve accuracy. sEV instead penalises only revisions that move a forecast *away* from the truth, or that overshoot it, rewarding accuracy-improving revisions while separating them from harmful volatility:

$$\mathrm{EV}(y,\;\hat{\mathbf{y}}_1,\; \hat{\mathbf{y}}_2) = \mathrm{QL}(\hat{\mathbf{y}}_2,\hat{\mathbf{y}}_1) - \big(\mathrm{QL}(y,\hat{\mathbf{y}}_1)-\mathrm{QL}(y,\hat{\mathbf{y}}_2)\big)$$

$$\mathrm{sEV} = \frac{\sum_{b,t,h}\mathrm{EV}\big(y_{b,t,h},\;\hat{\mathbf{y}}_{b,t,h+1},\;\hat{\mathbf{y}}_{b,t+1,h}\big)}{\sum_{b,t,h}|y_{b,t,h}|}$$

where $\mathrm{QL}$ is the quantile loss at level $q \in \mathcal{Q}=\{0.1, 0.2, \dots, 0.9\}$:

$$\mathrm{QL}_q(y, \hat{y}^{(q)}) = q(y-\hat{y}^{(q)})_+ + (1-q)(\hat{y}^{(q)}-y)_+$$

The first EV term is what the revision cost; the bracketed term is the accuracy it bought. Because $\mathrm{QL}$ obeys the triangle inequality the second can never exceed the first, so $\mathrm{EV} \geq 0$. It is zero for a revision that moves straight toward the truth without overshooting it, and positive for one that moves away from the truth or overshoots it. Anything above zero is revision that was not matched by an equal gain in accuracy.

### Which call gives which

| metric | call |
|:--|:--|
| sFPC | `forecast_percentage_change(preds=..., symmetric=True)` |
| FPC, one-sided | `forecast_percentage_change(preds=..., symmetric=False)` |
| sEV | `excess_volatility(targets=..., preds=..., quantiles=..., scaling=True)` |
| EV, unscaled | `excess_volatility(targets=..., preds=..., quantiles=..., scaling=False)` |

Both implementations add a small `eps` to the denominator as a division guard, which the formulas above omit.

Both metrics need **overlapping** windows. If the step between FCDs is greater than or equal to the horizon, each date is forecast only once and there is no revision to measure.

## Setup

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

sys.path.append(os.path.abspath(os.path.join("..", "tsforge")))
from metrics.eval_losses import crps, mae
from metrics.stability_metrics import excess_volatility, forecast_percentage_change

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.family"] = "serif"

## Two forecasters that accuracy cannot tell apart

To see why a volatility metric is needed at all, we build two forecasters by hand and give them the *same* accuracy.

The series is a week of hourly demand with a daily shape and a weekday/weekend effect. We forecast it from six FCDs spaced 24 hours apart, each with a 48 hour horizon, so every window overlaps the next by 24 hours.

In [ ]:
HORIZON = 48
STEP_SIZE = 24
N_HOURS = 168
FCDS = [0, 24, 48, 72, 96, 120]

QUANTILES = [0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9]
BAND = 3.5
# half-width of each quantile relative to the median, as a multiple of BAND
SPREAD = {0.1: 1.0, 0.25: 0.55, 0.4: 0.25, 0.5: 0.0, 0.6: -0.25, 0.75: -0.55, 0.9: -1.0}

hours = np.arange(N_HOURS)


def daily_shape(h):
    t = (h % 24) / 24
    return (
        40
        + 18 * np.sin(np.pi * (t - 0.05))
        + 10 * np.exp(-((t - 0.83) ** 2) / 0.01)
        - 6 * np.exp(-((t - 0.15) ** 2) / 0.008)
    )


weekday_effect = np.array([1.0, 1.02, 1.03, 1.02, 1.04, 0.92, 0.88])[hours // 24]
truth = daily_shape(hours) * weekday_effect + np.random.default_rng(7).normal(0, 0.5, N_HOURS)

The two forecasters differ only in **where their error comes from**.

The *steady* forecaster is wrong in a way that persists: it draws one error signal for the whole week, so every FCD is wrong about a given date in the same direction. Its forecasts for a shared date barely move.

The *erratic* forecaster draws fresh, independent error at every FCD, and additionally flips the sign of a shift on the overlapping half of each window. Its forecasts for a shared date swing back and forth.

In [ ]:
def build_quantiles(medians):
    """Stack per-FCD medians into the [B, T, H, C] and [B, T, H, C, Q] arrays the metrics take."""
    y = np.stack([truth[s : s + HORIZON] for s in FCDS])[None, :, :, None]
    y_hat = np.empty((1, len(FCDS), HORIZON, 1, len(QUANTILES)))
    for i, q in enumerate(QUANTILES):
        y_hat[0, :, :, 0, i] = np.stack(medians) - SPREAD[q] * BAND
    return y, y_hat


# steady: one persistent error signal shared by every FCD, plus a little jitter
rng = np.random.default_rng(13)
persistent_error = rng.normal(0, 2.65, N_HOURS)
steady_medians = [
    truth[s : s + HORIZON] + persistent_error[s : s + HORIZON] + rng.normal(0, 0.8, HORIZON)
    for s in FCDS
]

# erratic: fresh error each FCD, plus a sign-flipping shift on the overlapping half
rng = np.random.default_rng(77)
erratic_medians = []
for i, s in enumerate(FCDS):
    shift = np.zeros(HORIZON)
    shift[HORIZON // 2 :] = (-1) ** i * 3.0
    erratic_medians.append(truth[s : s + HORIZON] + rng.normal(0, 2.0, HORIZON) + shift)

y_steady, y_hat_steady = build_quantiles(steady_medians)
y_erratic, y_hat_erratic = build_quantiles(erratic_medians)

Plotting the six forecast windows against the truth makes the difference obvious.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, sharey=True)
colors = plt.cm.viridis(np.linspace(0, 0.9, len(FCDS)))

for ax, medians, title in [
    (axes[0], steady_medians, "Steady forecaster"),
    (axes[1], erratic_medians, "Erratic forecaster"),
]:
    ax.plot(hours, truth, color="black", linewidth=1.2, label="actual", zorder=5)
    for i, (start, median) in enumerate(zip(FCDS, medians)):
        window = np.arange(start, start + HORIZON)
        ax.plot(window, median, color=colors[i], linewidth=1.6, alpha=0.9)
        ax.fill_between(window, median - BAND, median + BAND, color=colors[i], alpha=0.12, linewidth=0)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_ylabel("demand")

axes[0].legend(loc="upper right", fontsize=9)
axes[1].set_xlabel("hour")
plt.tight_layout()
plt.show()

Now score them. First on accuracy:

In [ ]:
for name, y, y_hat in [("steady", y_steady, y_hat_steady), ("erratic", y_erratic, y_hat_erratic)]:
    median = y_hat[..., QUANTILES.index(0.5)]
    print(
        f"{name:8s}  MAE={mae(preds=median, targets=y):6.3f}  "
        f"sCRPS={crps(preds=y_hat, targets=y, quantiles=QUANTILES, scale=True):6.4f}"
    )

The two are effectively tied: any accuracy-based comparison would call this a wash. Now the volatility metrics:

In [ ]:
for name, y, y_hat in [("steady", y_steady, y_hat_steady), ("erratic", y_erratic, y_hat_erratic)]:
    ev = excess_volatility(
        targets=y, preds=y_hat, quantiles=QUANTILES, stride=STEP_SIZE, scaling=True
    )
    sfpc = forecast_percentage_change(
        preds=y_hat[..., QUANTILES.index(0.5)], stride=STEP_SIZE
    )
    print(f"{name:8s}  sEV={ev:7.4f}  sFPC={sfpc:6.3f}")

The erratic forecaster revises its predictions several times as much as the steady one, and its scaled excess volatility is roughly double: the revisions it makes are not buying it accuracy. Neither fact is visible in `MAE` or `sCRPS`.

This is the case for tracking volatility alongside accuracy rather than instead of it. The two measure different things, and a model can be good at one and bad at the other. Lower is better on both: the goal is to cut volatility without giving up accuracy.

## References

- Willa Potosnak, Malcolm Wolff, Mengfei Cao, Ruijun Ma, Tatiana Konstantinova, Dmitry Efimov, Michael W. Mahoney, Boris Oreshkin, Kin G. Olivares. [Forking-Sequences: Statistically and Computationally Efficient Multi-Horizon Forecasting with Reduced Volatility](https://openreview.net/forum?id=dXdycy7WCX). Transactions on Machine Learning Research (2026).